## Set up

**libraries**

In [ ]:
import json
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import Adam
from torch.nn import Module, Sequential, Linear, LSTM
from torch.nn.utils import clip_grad_norm_
from torch.nn.functional import mse_loss
from sklearn.preprocessing import MinMaxScaler
from torch import Tensor
from typing import Iterator
Loader = Iterator[Tensor]

**Constants**

In [ ]:
YSIZE  = 12
GROUP = ["doc", "tressure group"]
VAL   = "volume"
YEAR  = "year"
MONTH = "month"

**target**

Which year to forecast and from which point within the year

In [ ]:
PYEAR  = 2025
SLEN   = 0
TRAIN  = True

**Training parameters**

In [ ]:
EPOCHS = 170
HIDNS  = 450
BSIZE  = 4
THRESH = 1e+6
LRATE  = 1e-3
LASSO  = 1e-2
NOISE  = 1e-2

**paths**

In [ ]:
FOLDER                = "../../data/deep learning project data/2025/coin 1/"
DATA_PATH             = "../../data/deep learning project data/documents coin 1.csv"
OUTPUT_PATH_CURR_YEAR = FOLDER + f"results {PYEAR} from {SLEN}.csv"
OUTPUT_PATH_NEXT_YEAR = FOLDER + f"results {PYEAR + 1}.csv"
MODEL_PATH            = FOLDER + "model.pt"
SPECS_PATH            = FOLDER + "parameters.json"

<br>
<br>

### Organizing the data

In [ ]:
data = pd.read_csv(DATA_PATH, usecols = [ * GROUP, YEAR, MONTH, VAL])

filter years beyond the target

In [ ]:
data = data[data[YEAR].between(2012, PYEAR)]

filter empty series

In [ ]:
data = data.groupby(GROUP).filter(lambda g : g.groupby(YEAR)[VAL].sum().mean() > THRESH)

Setting groups as features <br>
Each group has its own time series

In [ ]:
data = data.pivot_table(index = [YEAR, MONTH], columns = GROUP, values = VAL, aggfunc = 'sum', fill_value = 0).sort_index()

In [ ]:
train = data.loc[data.index.get_level_values(level = YEAR) != PYEAR]
test  = data.loc[data.index.get_level_values(level = YEAR) >= PYEAR - 1]

Filtering the training data for full years only

In [ ]:
train = train.groupby(level = YEAR).filter(lambda g : g.count().min() == YSIZE)

Min-Max normalizing each group. Preserves order and sets a [0, 1] range.

In [ ]:
scaler = MinMaxScaler()
scaler.fit(train);

Making a tensor data set for RNN training. <br>

* batch - year
* squence step - month
* feature - group

Actually, each batch will be made of 2 consequtive years. <br>
So that the network could infer from past year data to current one. 

In [ ]:
def to_tensor(table: pd.DataFrame) -> Tensor:
    table = table.groupby(level = YEAR)
    table = table.apply(scaler.transform)
    table = table.apply(torch.FloatTensor)
    table = table.values.tolist()
    table = torch.stack([torch.cat([prev, curr]) for prev, curr in zip(table, table[1:])])
    return table

In [ ]:
train_ten = to_tensor(train)
test_ten  = to_tensor(test)

In [ ]:
target_ten = train_ten[:, 1:, :]
train_ten  = train_ten[:, :-1, :]

In [ ]:
assert train_ten.size(2) == target_ten.size(2) == test_ten.size(2), "number of series missmatch"
assert train_ten.size(1) == target_ten.size(1) == 2 * YSIZE - 1,    "series length missmatch"
assert train_ten.size(0) == target_ten.size(0),                     "number of years missmatch"
assert test_ten.size(0) == 1,                                       "test set should contain only one year"
assert SLEN <= test_ten.size(1) - YSIZE,                            "seed length exeeds max month"

In [ ]:
tdataset = TensorDataset(train_ten, target_ten)
loader : Loader = DataLoader(tdataset, batch_size = BSIZE, shuffle = True)

<br>

### Building architectures

In [ ]:
class Forecaster(Module):
    def fit(self, loader: Loader, epochs: int, lr: float): pass
    def forecast(self, seed: Tensor, fh: int): pass

In [ ]:
class BiYearlyRNN(Forecaster):

    def __init__(self, input_dim: int, lstm_hid: int):
        super().__init__()
        
        self.rnn = LSTM(input_dim, lstm_hid, batch_first = True)
        self.head = Sequential(Linear(lstm_hid, input_dim))


    def forward(self, x: Tensor, hid = None) -> tuple[Tensor, Tensor]:
        x, hid = self.rnn(x, hid)
        x = self.head(x)
        return x, hid
    

    def lasso(self) -> Tensor:
        return torch.cat([p.abs().flatten() for n, p in self.named_parameters() if "weight_ih" in n or "linear" in n]).mean() * LASSO
    

    def loss(self, pred: Tensor, target: Tensor) -> Tensor:
        return mse_loss(pred, target) + self.lasso()


    def fit(self, loader: Loader, epochs: int, lr: float):
        
        optimizer = Adam(self.parameters(), lr=lr)
        self.train()


        def train_step(data_batch: Tensor, target_batch: Tensor) -> float:
            
            data_batch += torch.randn_like(data_batch) * NOISE
            optimizer.zero_grad()
            curr_year_preds, _ = self.forward(data_batch)
            loss = self.loss(curr_year_preds, target_batch)
            loss.backward()
            clip_grad_norm_(self.parameters(), max_norm = 1.0)
            optimizer.step()
            return loss.item()


        for epoch in range(1, epochs + 1):
            
            total_loss = sum(train_step(data_batch, target_batch) for data_batch, target_batch in loader)
            avg_loss = total_loss / len(loader)

            if epoch % 10 == 0: print(f"Epoch {epoch:3d} | Avg Loss: {avg_loss:.4f}")


    @torch.no_grad
    def forecast(self, seed: Tensor, fh: int, hid = None):
        
        self.eval()
        forecast = []

        for step in range(seed.size(1)):
            val = seed[:, step, :]
            forecast.append(val)
            _, hid = self.forward(val, hid)

        for step in range(fh):
            val, hid = self.forward(val, hid)
            forecast.append(val)

        return torch.concat(forecast).relu()

In [ ]:
model = BiYearlyRNN(input_dim = train_ten.size(-1), lstm_hid = HIDNS)

<br>

### Training

In [ ]:
if TRAIN:

    model.fit(loader, EPOCHS, LRATE)
    torch.save(model.state_dict(), MODEL_PATH)

else:

    try: model.load_state_dict(torch.load(MODEL_PATH))
    except FileNotFoundError: raise FileNotFoundError("Model file not found. Please train the model first.")
    except Exception as e:    raise RuntimeError(f"Error loading model. Might be model with different architecture from an older run. details: {e}")

<br>

### Forecasting

In [ ]:
curr_year_pred = model.forecast(test_ten[:, : YSIZE + SLEN, :], YSIZE - SLEN)
curr_year_pred = curr_year_pred.unsqueeze(0)
next_year_pred = model.forecast(curr_year_pred[:, : YSIZE, :], YSIZE)

In [ ]:
def organize_forecast(forecast: Tensor, test: pd.DataFrame) -> pd.DataFrame:

    # dimention check
    forecast = forecast.squeeze()
    assert forecast.size(0) == YSIZE * 2,     "forecast length mismatch"
    assert forecast.size(1) == test.shape[1], "forecast series count mismatch"

    # convertion
    forecast = forecast.detach().numpy()

    # denormalization
    forecast = scaler.inverse_transform(forecast)

    # cat out the warm up
    forecast = forecast[YSIZE:]
    test = test.iloc[YSIZE:]

    # tabularizing results
    test     : pd.DataFrame = test.transpose()
    forecast : pd.DataFrame = pd.DataFrame(index = test.index, data = forecast.transpose())

    test = test.sum(axis = 1).to_frame(name = "actual")
    forecast = forecast.sum(axis = 1).to_frame(name = "forecast")
    results = forecast.join(test)

    # error stats
    results.sort_values(by = "actual", inplace = True, ascending = False)
    results.loc[results["actual"] == 0, "forecast"] = 0
    results["abs err"] = results["forecast"].sub(results["actual"]).abs()
    results["rel err"] = results["abs err"].div(results["actual"]).round(2)


    results = results.reset_index()

    return results

In [ ]:
organize_forecast(curr_year_pred, test).to_csv(OUTPUT_PATH_CURR_YEAR, index = False)
organize_forecast(next_year_pred, test).to_csv(OUTPUT_PATH_NEXT_YEAR, index = False)

Saving the runs parametes for future refrence 

In [ ]:
with open(SPECS_PATH, 'w') as file:

    specs = {
        "PYEAR"  : PYEAR,
        "EPOCHS" : EPOCHS,
        "HIDNS"  : HIDNS,
        "BSIZE"  : BSIZE,
        "SLEN"   : SLEN,
        "LRATE"  : LRATE,
        "THRESH" : THRESH,
        "LASSO"  : LASSO,
        "NOISE"  : NOISE
    }
    json.dump(specs, file, indent = 4)